In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
import copy
import matplotlib.pyplot as plt
from munch import Munch
import itertools


from ICML_comparisons import generate_random_hmm, find_longest_consecutive_sequence
from ICML_MVRseq import Viterbi_torch_list, create_seqconstraint
# from ICML_beam_search import *
from helper import sample_hmm

In [3]:
import gc
# Clear all cached memory
torch.cuda.empty_cache()

# Force Python garbage collection
gc.collect()

# Clear cache again after garbage collection
torch.cuda.empty_cache()

# 1. HMM Objects

1. HMM's can be represented by its update matrix, emissions matrix, and initial vector. ```generate_random_hmm``` will create a list of ```torch.Tensors``` in the above order, with each tensor sampled randomly and normalized. 

2. These tensors are then converted into dictionaries, whose keys are ```hidden_states``` (transition, initial) as well as ```emit_states``` (emission). For example,  the **value**```hmm_emit[(0,1)]``` of the **key** ```(0,1)``` is $P(Y= 1 | X = 0)$, where $X,Y$ are the hidden, emission states respectively.

3. The above dictionaries are then wrapped into a ```Munch``` object. Munch is basically a dictionary, except you can call attributes (ie. keys) via ```munch.attribute``` instead of ```munch['attribute']```

In [4]:
# Generate random HMM
print("=== Generating Random HMM ===")

#Specify size of hidden/emission space
num_states = 50
num_emissions = 10

hmm_params = generate_random_hmm(num_states, num_emissions, seed=42)
transition_matrix, emission_matrix, initial_vector = hmm_params

print(f"Generated HMM with {num_states} states and {num_emissions} emissions")

hidden_states = list(range(num_states))
emit_states = list(range(num_emissions))
hidden_size, emit_size = len(hidden_states), len(emit_states)


hmm_transition = {}
for i in range(hidden_size):
    for j in range(hidden_size):
        hmm_transition[hidden_states[i],hidden_states[j]] = transition_matrix[i,j].item()

hmm_emit = {}
for i in range(hidden_size):
    for j in range(emit_size):
        hmm_emit[hidden_states[i],emit_states[j]] = emission_matrix[i,j].item()
        
hmm_startprob = {}
for i in range(hidden_size):
    hmm_startprob[hidden_states[i]] = initial_vector[i]

hmm = copy.deepcopy(Munch(states = hidden_states, emits = emit_states, tprob = hmm_transition, eprob = hmm_emit, initprob = hmm_startprob))

=== Generating Random HMM ===
Generated HMM with 50 states and 10 emissions


# 2. Constraint Objects

1. Constraints are also ```Munch``` objects as well.
2. Below is an example of a constraint object. Each attribute is a Boolean function that correspond to the update/init/eval functions of a DFA:
    1. ```update_fun(k,r,r_past)```: In state ```r_past``` and we encounter hidden state ```k```, return True if we transition to state ```r```. Otherwise False.
    2. ```init_fun(k,r)```: Returns True if hidden state ```k``` intializes the mediation variable to ```r```. 
    3. ```eval(k,r)```: Shouldn't depend on ```k```, but is a legacy of older code. Return True if ```r``` is an accepting state.


In [5]:
def update_fun(k , r, r_past):
    '''
    r =  [True,False] x [0,1,2]
    r[0] is True if previoius state was NOT dis.
    counts number of entries into dis (ie. number of dis regions)
    '''
    prev, count = r_past

    if prev and k == 'dis':
        count = min(count + 1, 2)    
    
    return r == (k != 'dis',count) 

def init_fun(k, r):

    if k == 'dis':
        return r == (False,1)
    else:
        return r == (True,0)

def eval_fun(k,r):
    return r[1] == 1 #must be exactly 1.

m_states = list(itertools.product([True,False], list(range(3))))


disvisit_cst = copy.deepcopy(Munch(update_fun = update_fun, init_fun = init_fun, eval_fun = eval_fun, m_states = m_states))

## 2.a Another Constraint Example

Here is the subsequence constraint generator  ```create_seqconstraint``` from the scaling experiments. You provide the size of the hidden space ```num_states``` and the length of the subsequence. Will create a constraint object - a Munch class with update_fun, init_fun, eval_fun, m_states attributes - the reflects that subsequence constraint. 

In [6]:
# Define constraint: encourage consecutive sequence [0,1,2,...,N-1]
constraint_length = 10
print(f"\n=== Constraint: Consecutive Sequence [0,1,2,...,{constraint_length-1}] ===")
required_transitions = [(i, i+1) for i in range(constraint_length - 1)]
print(f"Required transitions: {required_transitions}")

#Create constraint object for MVR
seq_cst = create_seqconstraint(num_states,constraint_length)
cst_list = [seq_cst]


=== Constraint: Consecutive Sequence [0,1,2,...,9] ===
Required transitions: [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9)]


# 3. Sampling from the HMM

There are quite a few functions scattered in the notebook that will draw a hidden/emission sequence from an HMM. If you want to draw a feasible path, the most straightforward way to do it is sample and reject.

1. ```sample_hmm``` (run below) is a generic function in the ```helper.py``` file. You provide the hmm, time horizon, and an optional constraint checker that checks for feasibilty.
2. There are other bespoke functions that were tailored for their applications:
   1. ```single_simulation``` in the dna notebook
   2. ```sample_from_hmm``` in the ICML_comparisons file. 

In [7]:
# Sample from the HMM
print("\n=== Unconstrained Sample from HMM ===")
states, observations = sample_hmm(hmm,100)
print(f' First ten hidden states/emissions: {list(zip(states, observations))[:10]}')


=== Unconstrained Sample from HMM ===
 First ten hidden states/emissions: [(21, 3), (25, 2), (49, 6), (19, 7), (10, 9), (31, 5), (0, 7), (17, 6), (1, 5), (31, 5)]


In [8]:
# Here's a peverse example of a constraint_checker function that randomly returns True 10% of the time.

import random
def randomly_feasible(x):
    flip = random.random()
    if flip > .9:
        return True
    else:
        return False

print("\n=== Constrained Sample from HMM ===")

states2, observations2 = sample_hmm(hmm, 100, randomly_feasible)

print(f' First ten hidden states/emissions: {list(zip(states2, observations2))[:10]}')


=== Constrained Sample from HMM ===
 First ten hidden states/emissions: [(19, 7), (23, 2), (5, 3), (35, 8), (32, 2), (17, 2), (3, 0), (11, 0), (14, 7), (16, 1)]


# 4. Running Constrained Inference

Use ```Vitberi_torch_list``` to run constrained inference. It's the same function we previously discussed. Here, I'm using the subsequence constraint from ICML_comparisonStudies.

In [9]:
seq_cst = create_seqconstraint(num_states,constraint_length)
cst_list = [seq_cst]
# cst_parameters = create_seq_cst_params(num_states, constraint_length, device='cpu')
# input_list = Viterbi_preprocess(hmm, cst_list, observations, dtype = torch.float32,  device = 'cuda:0', debug = False, hmm_params = [transition_matrix,initial_vector], cst_params = cst_parameters)


In [10]:
opt_state_list, opt_augstateix_list = Viterbi_torch_list(hmm, cst_list, observations, dtype = torch.float32,  device = 'cuda:0', debug = False, num_corr = 0)


In [14]:
print(f'Comparison of Ground-Truth vs Inferred States\n Ground Truth: {states[:20]} \n Inferred: {opt_state_list[:20]} \n Inferred Sequence Feasibility: \
{find_longest_consecutive_sequence(opt_state_list) == constraint_length}')

Comparison of Ground-Truth vs Inferred States
 Ground Truth: [21, 25, 49, 19, 10, 31, 0, 17, 1, 31, 4, 26, 40, 0, 36, 6, 36, 8, 44, 16] 
 Inferred: [48, 17, 18, 22, 35, 1, 16, 27, 26, 41, 14, 23, 24, 11, 33, 32, 38, 22, 35, 30] 
 Inferred Sequence Feasibility: True
